# 🧬 Drug-Protein Interaction — Predicción sobre Librería Nueva

**Modelo:** XGBoost + Morgan FP (ECFP4, 2048 bits) + Descriptores fisicoquímicos  
**Entrenado en:** BindingDB (109,028 moléculas, scaffold split validado)  
**Umbral de actividad:** Ki/IC50/Kd < 1000 nM → BINDER

## ¿Qué necesitas?
1. El modelo entrenado: `xgboost.pkl` (generado en el notebook principal)
2. Tu librería de compuestos: un CSV con una columna llamada `smiles`

## ¿Qué obtienes?
- Probabilidad de binding para cada compuesto
- Clasificación BINDER / NO BINDER
- Métricas de rendimiento (si tienes etiquetas reales)
- Visualizaciones: top hits, espacio químico, distribución de scores

## 0. INSTALACIÓN

In [ ]:
!pip install rdkit xgboost scikit-learn joblib matplotlib seaborn -q

## 1. CONFIGURACIÓN — Edita solo esta celda

In [ ]:
from pathlib import Path

# ── Rutas — compatible con Colab y local ────
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/DrugProtein_ML')
except Exception:
    PROJECT_DIR = Path.cwd()

MODEL_PATH = PROJECT_DIR / 'xgboost.pkl'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'predictions'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# ── EDITA AQUÍ — ruta a tu CSV ──
# El CSV debe tener una columna 'smiles'
# Opcionalmente una columna 'label' (0/1) para obtener métricas de rendimiento
INPUT_CSV = PROJECT_DIR / 'mi_libreria.csv'

# Verificar modelo
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Modelo no encontrado en outputs/models/xgboost.pkl\n"
        "Ejecuta primero el notebook de entrenamiento."
    )
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        "CSV no encontrado. Edita INPUT_CSV con la ruta a tu librería."
    )

print(f"✅ Modelo encontrado: {MODEL_PATH}")
print(f"✅ CSV encontrado: {INPUT_CSV}")

# ── Parámetros ──────
THRESHOLD     = 0.5
MORGAN_RADIUS = 2
MORGAN_BITS   = 2048

print(f"✅ Config lista | Threshold: {THRESHOLD}")

## 2. IMPORTS Y CARGA DEL MODELO

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Lipinski, QED, Draw
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


RDLogger.DisableLog('rdApp.*')
sns.set_style('whitegrid')
np.random.seed(42)

# Cargar modelo
assert MODEL_PATH.exists(), f'Modelo no encontrado en {MODEL_PATH}. Ejecuta primero el notebook de entrenamiento.'
model = joblib.load(MODEL_PATH)
print(f'✅ Modelo cargado: {MODEL_PATH.name}')
print(f'   Tipo: {type(model).__name__}')
print(f'   Features esperados: {model.n_features_in_}')

## 3. PIPELINE DE PREPROCESADO Y FEATURIZACIÓN

In [ ]:
remover    = SaltRemover()
morgan_gen = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)

def clean_smiles(s):
    """Canonicaliza SMILES: desala, toma fragmento mayor, sanitiza."""
    try:
        mol = Chem.MolFromSmiles(str(s))
        if not mol: return None
        mol = remover.StripMol(mol, dontRemoveEverything=True)
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
        if len(frags) > 1:
            mol = max(frags, key=lambda m: m.GetNumAtoms())
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def featurize(smiles):
    """Morgan FP (2048) + 10 descriptores fisicoquímicos."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    fp  = morgan_gen.GetFingerprint(mol)
    fp_arr = np.frombuffer(fp.ToBitString().encode(), dtype=np.uint8) - ord('0')
    desc = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumHDonors(mol),
        Descriptors.TPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Lipinski.NumAromaticRings(mol),
        QED.qed(mol),
        mol.GetNumHeavyAtoms(),
        Descriptors.FractionCSP3(mol)
    ], dtype=np.float32)
    return np.concatenate([fp_arr.astype(np.float32), desc])

print('✅ Pipeline de preprocesado listo')

## 4. CARGA Y PROCESADO DE LA LIBRERÍA

In [ ]:
print('=' * 60)
print('CARGANDO LIBRERÍA')
print('=' * 60)

if INPUT_CSV.exists():
    df = pd.read_csv(INPUT_CSV)
    print(f'CSV cargado: {len(df):,} compuestos')
    if 'smiles' not in df.columns:
        raise ValueError('El CSV debe tener una columna llamada "smiles"')

has_labels = 'label' in df.columns
print(f'Etiquetas reales disponibles: {"Si" if has_labels else "No — solo prediccion"}')

# Limpiar SMILES
print('Limpiando SMILES...')
df['smiles_clean'] = df['smiles'].apply(clean_smiles)
n_invalid = df['smiles_clean'].isna().sum()
if n_invalid > 0:
    print(f'Advertencia: {n_invalid} SMILES invalidos descartados')
df = df.dropna(subset=['smiles_clean']).reset_index(drop=True)

# Featurizar
print('Featurizando...')
features, valid_idx = [], []
for i, row in df.iterrows():
    f = featurize(row['smiles_clean'])
    if f is not None:
        features.append(f)
        valid_idx.append(i)

df = df.loc[valid_idx].reset_index(drop=True)
X  = np.array(features)

print(f'Listos para predecir: {len(df):,} compuestos')
print(f'Feature matrix: {X.shape}')

## 5. PREDICCIÓN

In [ ]:
print('=' * 60)
print('PREDICIENDO')
print('=' * 60)

df['prob_binder']  = model.predict_proba(X)[:, 1]
df['prediction']   = (df['prob_binder'] >= THRESHOLD).map({True: 'BINDER', False: 'NO BINDER'})
df['confidence']   = df['prob_binder'].apply(
    lambda p: 'Alta' if (p > 0.75 or p < 0.25) else 'Media'
)

n_binders    = (df['prediction'] == 'BINDER').sum()
n_nobinders  = (df['prediction'] == 'NO BINDER').sum()
n_high_conf  = (df['confidence'] == 'Alta').sum()

print(f'\nResumen de predicciones:')
print(f'   BINDER:      {n_binders:,} ({n_binders/len(df)*100:.1f}%)')
print(f'   NO BINDER:   {n_nobinders:,} ({n_nobinders/len(df)*100:.1f}%)')
print(f'   Alta confianza: {n_high_conf:,} ({n_high_conf/len(df)*100:.1f}%)')

print(f'\nTabla de resultados:')
cols_show = ['smiles_clean', 'prob_binder', 'prediction', 'confidence']
if has_labels:
    cols_show.insert(2, 'label')
print(df[cols_show].sort_values('prob_binder', ascending=False).to_string(index=False))

# Guardar CSV
out_csv = OUTPUT_DIR / 'predictions.csv'
df.to_csv(out_csv, index=False)
print(f'\nResultados guardados en {out_csv}')

## 6. MÉTRICAS DE RENDIMIENTO (solo si tienes etiquetas reales)

In [ ]:
if not has_labels:
    print('Sin etiquetas reales — saltando métricas de rendimiento.')
    print('   Añade una columna "label" (0/1) a tu CSV para obtener métricas.')
else:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, f1_score,
        matthews_corrcoef, confusion_matrix, roc_curve, precision_recall_curve
    )

    y_true = df['label'].values
    y_prob = df['prob_binder'].values
    y_pred = (y_prob >= THRESHOLD).astype(int)

    metrics = {
        'ROC-AUC':  roc_auc_score(y_true, y_prob),
        'PR-AUC':   average_precision_score(y_true, y_prob),
        'F1':       f1_score(y_true, y_pred),
        'MCC':      matthews_corrcoef(y_true, y_pred),
    }

    print('=' * 60)
    print('MÉTRICAS DE RENDIMIENTO')
    print('=' * 60)
    for k, v in metrics.items():
        print(f'   {k}: {v:.4f}')

    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['NO BINDER', 'BINDER'],
                yticklabels=['NO BINDER', 'BINDER'])
    axes[0].set_title('Matriz de Confusión', fontweight='bold')
    axes[0].set_ylabel('Real')
    axes[0].set_xlabel('Predicho')

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, color='#2196F3', lw=2,
                 label='AUC = {:.3f}'.format(metrics['ROC-AUC']))
    axes[1].plot([0,1],[0,1],'k--', alpha=0.3)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('Curva ROC', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[2].plot(rec, prec, color='#FF5722', lw=2,
                 label='AP = {:.3f}'.format(metrics['PR-AUC']))
    axes[2].axhline(y_true.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Curva Precision-Recall', fontweight='bold')
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. VISUALIZACIONES

In [ ]:
print('=' * 60)
print('VISUALIZACIONES')
print('=' * 60)

def mol_safe(s):
    try: return Chem.MolFromSmiles(s)
    except: return None

# ── VIZ 1: Distribución de scores ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['prob_binder'], bins=40, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', lw=2, label='Threshold ({})'.format(THRESHOLD))
axes[0].set_xlabel('Probabilidad de Binding')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Scores', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

counts = df['prediction'].value_counts()
colors_pie = ['#2196F3', '#FF5722']
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción BINDER / NO BINDER', fontweight='bold')

plt.suptitle('Resumen de Predicciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── VIZ 2: Top hits — moléculas con mayor probabilidad ─────────
top_hits = df.nlargest(min(10, len(df)), 'prob_binder')
mols_top = [mol_safe(s) for s in top_hits['smiles_clean']]
mols_top = [m for m in mols_top if m is not None]

if mols_top:
    labels_top = ['Score: {:.3f}\n{}'.format(p, pred)
                  for p, pred in zip(top_hits['prob_binder'], top_hits['prediction'])]
    img = Draw.MolsToGridImage(
        mols_top, molsPerRow=5, subImgSize=(300, 250),
        legends=labels_top, returnPNG=False
    )
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Top Hits — Mayor probabilidad de binding', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'top_hits.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── VIZ 3: t-SNE (solo si hay suficientes moléculas) ─────────────────
if len(df) >= 20:
    from sklearn.manifold import TSNE
    print('\n📊 Generando t-SNE (puede tardar unos minutos)...')
    perp = min(30, len(df) // 3)
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    coords = tsne.fit_transform(X[:, :MORGAN_BITS])

    fig, ax = plt.subplots(figsize=(10, 8))
    for label, color in [('BINDER', '#2196F3'), ('NO BINDER', '#FF5722')]:
        mask = df['prediction'] == label
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=color, label=label, s=60, alpha=0.7, edgecolors='white', linewidth=0.5)

    # Anotar los top 3 hits
    top3_idx = df['prob_binder'].nlargest(3).index
    for i in top3_idx:
        ax.annotate('#{} ({:.2f})'.format(i+1, df.loc[i, 'prob_binder']),
                    (coords[i, 0], coords[i, 1]),
                    textcoords='offset points', xytext=(8, 4), fontsize=8,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.7))

    ax.set_title('t-SNE — Espacio químico de la librería', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(markerscale=1.5)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'tsne_library.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('t-SNE requiere al menos 20 moléculas.')

print('\n✅ Visualizaciones guardadas en {}'.format(OUTPUT_DIR))

## 8. REPORTE DE INFERENCIA

In [ ]:
import datetime

report = """
======================================================================
DRUG-PROTEIN INTERACTION ML — REPORTE DE INFERENCIA
======================================================================
Fecha:          {fecha}
Modelo:         XGBoost + Morgan FP (ECFP4, 2048 bits)
Entrenado en:   BindingDB (109,028 moléculas, scaffold split)
Umbral:         {threshold} (BINDER si prob >= {threshold})

LIBRERÍA ANALIZADA:
  Compuestos entrada:    {n_entrada:,}
  Compuestos válidos:    {n_validos:,}
  BINDERS predichos:     {n_binders:,} ({pct_binders:.1f}%)
  NO BINDERS predichos:  {n_nobinders:,} ({pct_nobinders:.1f}%)
  Alta confianza:        {n_high_conf:,} ({pct_high:.1f}%)

TOP 5 HITS:
{top5}

ARCHIVOS GENERADOS:
  predictions.csv        → Tabla completa con scores
  score_distribution.png → Distribución de probabilidades
  top_hits.png           → Visualización molecular top hits
  tsne_library.png       → Mapa del espacio químico
======================================================================
""".format(
    fecha=datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    threshold=THRESHOLD,
    n_entrada=len(df),
    n_validos=len(df),
    n_binders=n_binders,
    pct_binders=n_binders/len(df)*100,
    n_nobinders=n_nobinders,
    pct_nobinders=n_nobinders/len(df)*100,
    n_high_conf=n_high_conf,
    pct_high=n_high_conf/len(df)*100,
    top5=df.nlargest(5, 'prob_binder')[['smiles_clean','prob_binder','prediction']].to_string(index=False)
)

print(report)
with open(OUTPUT_DIR / 'inference_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print('✅ Reporte guardado en {}'.format(OUTPUT_DIR / 'inference_report.txt'))